# Week 17, Day 5: Best, worst case and space

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w017d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def find(items, target):
    for i, x in enumerate(items):
        if x == target:
            return i + 1          # how many comparisons it took
    return len(items)

data = list(range(100))
print(find(data, 0), find(data, 50), find(data, 99), find(data, -1))

In [ ]:
def comparisons(items, target):
    for i, x in enumerate(items):
        if x == target:
            return i + 1
    return len(items)

n = 1000
data = list(range(n))
results = [comparisons(data, t) for t in data]
print("best", min(results), "worst", max(results), "average", sum(results) / n)

> **Common mistake.** The best case isn't "a small input". Cases are always about different inputs **of the same size n**. Saying "the best case is an empty list" misses the point: every algorithm is fast on an empty list.

> **Key idea.** Best, worst and average are about different inputs of the **same** size. Unless someone says otherwise, a Big-O is for the worst case.

In [ ]:
def largest(items):           # O(1) extra space
    best = items[0]
    for x in items:
        if x > best:
            best = x
    return best

def has_duplicate(items):     # O(n) extra space: the set can hold every item
    seen = set()
    for x in items:
        if x in seen:
            return True
        seen.add(x)
    return False

def doubled(items):           # O(n) extra space: a whole new list
    return [2 * x for x in items]

print(largest([3, 8, 2]), has_duplicate([1, 2, 1]), doubled([1, 2]))

In [ ]:
def total(items, i=0):
    if i == len(items):
        return 0
    return items[i] + total(items, i + 1)      # n frames waiting at the deepest point

print(total(list(range(500))))
try:
    total(list(range(100_000)))
except RecursionError:
    print("RecursionError: 100,000 frames is too many")

In [ ]:
def reverse_in_place(items):
    lo, hi = 0, len(items) - 1
    while lo < hi:
        items[lo], items[hi] = items[hi], items[lo]
        lo += 1
        hi -= 1

data = [1, 2, 3, 4, 5]
reverse_in_place(data)
print(data)

In [ ]:
import sys

items = []
last = sys.getsizeof(items)
for i in range(40):
    items.append(i)
    size = sys.getsizeof(items)
    if size != last:
        print(f"{len(items):>3} items: {size} bytes")
        last = size

### Exercise: Is it sorted?

Write `sorted_check(items)` that returns `(is_sorted, comparisons)`, comparing neighbours from the start and stopping at the first pair out of order. Then set `best` and `worst` to the number of comparisons in the best and worst case for a list of 10 items.

In [ ]:
best = ___
worst = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert sorted_check([1, 2, 3, 4]) == (True, 3)\n    assert sorted_check([2, 1, 3, 4]) == (False, 1)\n    assert sorted_check([1, 3, 2]) == (False, 2)\n    assert sorted_check([]) == (True, 0) and sorted_check([5]) == (True, 0)\n    assert best == 1 and worst == 9")

### Exercise: Average comparisons

Write `average_comparisons(n)` that returns the average number of comparisons a linear search makes over all n positions where the target could be (target present, each position equally likely), by actually running a counting search for each target in `list(range(n))`. Then write `formula(n)` that gives the same answer without a loop.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert average_comparisons(1) == 1 and average_comparisons(4) == 2.5\n    assert average_comparisons(100) == 50.5\n    for n in (1, 2, 7, 50, 1000):\n        assert formula(n) == average_comparisons(n)\n    assert \"for \" not in source.split(\"def formula\", 1)[1]")

### Exercise: How much space?

Give the **extra** space each function uses, as `"O(1)"`, `"O(log n)"` or `"O(n)"`, for a list of n items.

In [ ]:
def s1(items):
    count = 0
    for x in items:
        if x > 0:
            count += 1
    return count

def s2(items):
    return sorted(items)[0]

def s3(items):
    if not items:
        return 0
    return items[0] + s3(items[1:])

def s4(items, lo=0, hi=None):
    if hi is None:
        hi = len(items)
    if hi - lo <= 1:
        return items[lo] if hi > lo else 0
    mid = (lo + hi) // 2
    return max(s4(items, lo, mid), s4(items, mid, hi))

def s5(items):
    return len(set(items)) == len(items)

a1 = ___
a2 = ___
a3 = ___
a4 = ___
a5 = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a1 == \"O(1)\", \"s1\"\n    assert a2 == \"O(n)\", \"s2: sorted makes a new list\"\n    assert a3 == \"O(n)\", \"s3 (in fact worse: every slice is a copy, O(n^2) in total)\"\n    assert a4 == \"O(log n)\", \"s4: the stack is only log n deep, and no copies are made\"\n    assert a5 == \"O(n)\", \"s5\"")

### Exercise: Palindromes in O(1) space

Write `is_palindrome(text)` that checks whether a string reads the same backwards, using two positions moving towards each other, with O(1) extra space: no slicing, no `reversed`, no new strings or lists.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert is_palindrome(\"racecar\") and is_palindrome(\"abba\") and is_palindrome(\"\") and is_palindrome(\"x\")\n    assert not is_palindrome(\"ab\") and not is_palindrome(\"abca\")\n    assert is_palindrome(\"a\" * 100_000 + \"b\" + \"a\" * 100_000)\n    assert \"[::\" not in source and \"reversed\" not in source and \"join\" not in source")

### Exercise: Remove in place

Write `remove_all(items, value)` that removes every copy of `value` from the list **in place** in O(n) time and O(1) extra space, keeping the other items in order, and returns nothing. (Calling `items.remove(value)` in a loop is O(n²), since each removal shifts the items after it.) Copy each item you keep to the next free position at the front, then cut off the end with `del items[k:]`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    data = [3, 1, 3, 2, 3]\n    assert remove_all(data, 3) is None and data == [1, 2]\n    data = [1, 2]\n    remove_all(data, 9)\n    assert data == [1, 2]\n    data = [7] * 5\n    remove_all(data, 7)\n    assert data == []\n    big = [i % 3 for i in range(300_000)]\n    remove_all(big, 0)\n    assert len(big) == 200_000 and big[:4] == [1, 2, 1, 2]\n    assert \".remove(\" not in source and \"[x for\" not in source")

### Bug hunt: Too deep

`count_negatives` works on short lists but crashes with `RecursionError` on 10,000 items, because each call waits for the next: O(n) stack space. Rewrite it as a loop with O(1) extra space.

In [ ]:
def count_negatives(items, i=0):
    if i == len(items):
        return 0
    return (items[i] < 0) + count_negatives(items, i + 1)

print(count_negatives([-1, 2, -3]))
print(count_negatives([-1] * 10_000))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"both\"\n    expect_output(\"2\\n10000\")")

### Bug hunt: Never lucky

`contains` returns `(found, comparisons)`. It gives the right answer, but its best case is as slow as its worst: it always makes n comparisons. Fix it so it stops as soon as it finds the target.

In [ ]:
def contains(items, target):
    found = False
    comparisons = 0
    for x in items:
        comparisons += 1
        if x == target:
            found = True
    return found, comparisons

print(contains([5, 1, 9, 7], 5))
print(contains([5, 1, 9, 7], 8))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"stops early\"\n    expect_output(\"(True, 1)\\n(False, 4)\")")

### Puzzle: Reverse in place

Arrange the lines so `reverse_in_place` reverses a list using O(1) extra space. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Work these out without running them.

In [ ]:
a = ___   # comparisons in the best case for linear search of 50 items
b = ___   # extra space of `return items[::-1]` as a string
c = ___   # extra space of plain recursive fib(n) (the deepest the stack gets), as a string
d = ___   # True or False: an O(1)-space algorithm can still take O(n^2) time

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == 1 and b == \"O(n)\" and c == \"O(n)\" and d is True")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: rotate in place

Rotating a list right by k moves its last k items to the front: `[1, 2, 3, 4, 5]` rotated by 2 is `[4, 5, 1, 2, 3]`. `items[-k:] + items[:-k]` does it with O(n) extra space. Write `rotate(items, k)` that does it **in place** with O(1) extra space and O(n) time, and returns nothing, using the three-reversal trick: reverse the whole list, then reverse the first k items, then reverse the rest.

Write a helper `reverse_range(items, lo, hi)` that reverses `items[lo]` to `items[hi]` inclusive, in place. `k` may be bigger than the length (rotating by the length changes nothing), and the list may be empty.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    data = [1, 2, 3, 4, 5]\n    assert rotate(data, 2) is None and data == [4, 5, 1, 2, 3]\n    data = [1, 2, 3]\n    rotate(data, 3)\n    assert data == [1, 2, 3]\n    data = [1, 2, 3]\n    rotate(data, 7)\n    assert data == [3, 1, 2]\n    data = []\n    rotate(data, 4)\n    assert data == []\n    data = list(range(10))\n    reverse_range(data, 2, 5)\n    assert data == [0, 1, 5, 4, 3, 2, 6, 7, 8, 9]\ndef test_rules():\n    \"in place\"\n    big = list(range(200_000))\n    rotate(big, 1234)\n    assert big[0] == 200_000 - 1234 and big[1234] == 0\n    assert \"[::\" not in source and \"reversed\" not in source and \".reverse(\" not in source and \"+ items\" not in source")

## Warm-up

Same function, same size of input, and anything from 1 to 100 comparisons. "How fast is it?" has more than one answer.

## Best, worst and average

For an input of size n:

- the **best case** is the fewest steps the algorithm can take, on its most favourable input;
- the **worst case** is the most steps, on its least favourable input;
- the **average case** is the expected number of steps over all inputs, given some assumption about how likely each input is.

For a linear search through n items:

| Case | When | Comparisons | Big-O |
|---|---|---|---|
| Best | the target is first | 1 | O(1) |
| Worst | the target is last or missing | n | O(n) |
| Average | the target is present, equally likely to be anywhere | (n + 1)/2 | O(n) |

The average is about half the worst, but halving is a constant factor, so it's still O(n).

Another example: checking if a list is sorted by comparing neighbours stops at the first pair out of order. Best case: the first two items are out of order, 1 comparison. Worst case: the list **is** sorted, n − 1 comparisons.

## Which case to quote

When people say "linear search is O(n)", they mean the **worst case**, and that's usually the most useful thing to know:

- **It's a guarantee.** However unlucky the input, it won't be slower than this.
- **Unlucky inputs are common.** Searching for something that isn't there is the worst case for a search, and it happens all the time.
- **The average needs assumptions.** "Every position equally likely" is rarely exactly true. Real data has patterns.

The best case is rarely useful on its own: almost any algorithm can be lucky. The average case matters when the worst case is very rare and much slower. Quicksort (Week 19) is the famous example: O(n²) in the worst case, but O(n log n) on average, and fast in practice.

## Space

Every program also uses **memory**. The **space complexity** of an algorithm is how much **extra** memory it needs as n grows, not counting the input itself:

- **O(1) extra space:** a fixed number of variables, however big the input. Finding the largest item with a loop uses O(1): just `best` and the loop variable.
- **O(n) extra space:** something that grows with the input: a copy of the list, a set of items seen, a list of results.

Hidden costs apply to space too: `items[1:]`, `sorted(items)`, `list(...)` and `text.split()` all build new objects of size O(n).

**Recursion uses space.** Each call waiting on the call stack holds a frame with its own variables (Week 15). A recursion n calls deep uses O(n) space even if each call only has one variable. That's why recursion on long lists hits the recursion limit:

A loop does the same job with O(1) extra space. Divide and conquer recursion, which halves each time, is only O(log n) deep, so its stack space is small.

## Working in place

An algorithm that changes its input directly, using only O(1) extra space, works **in place**. Reversing a list with two positions moving towards each other is the classic example:

Compare `items[::-1]`, which is also O(n) time but builds a whole new list: O(n) extra space. `items.reverse()` and `items.sort()` work in place; `reversed(items)` and `sorted(items)` don't. In place saves memory, but the original is gone; that's the price.

You can see the memory a list itself uses with `sys.getsizeof`. Lists keep spare room, so `append` usually has a free slot and is O(1); now and then it has to move to a bigger block, but averaged over many appends (**amortised**) each one is still O(1):

## Trading space for time

Many speed-ups work by spending memory:

- **Memoisation** (Week 16) stores every answer: fib(n) drops from O(2ⁿ) time to O(n) time, and uses O(n) space for the memo.
- **A set or dictionary** beside a list turns O(n) lookups into O(1), at the cost of O(n) space: the duplicate check above is O(n) time and O(n) space, where comparing every pair is O(n²) time and O(1) space.
- **Precomputing** totals, such as running totals, makes later questions O(1) each.

The trade usually pays: memory is plentiful, and time is what users notice. But not always. A program that holds a set of every line in a 50 GB file will run out of memory, and on small devices memory is tight. Know both costs, then choose.

## Practice

## Quiz

## Daily challenge

## Before you go

For three functions you've written this week, write down the best case, the worst case and the extra space beside each. Tomorrow is the Forge: you'll take a slow program, measure it, analyse it, and make it fast.